# 3/4: Data preprocessing 2
By Niloufar Shahdoust (niloufar.shahdoust@utah.edu)

In [1]:
import os
import mat73
import numpy as np
import pandas as pd
import random
from matplotlib import cm
from matplotlib import colormaps
from ast import literal_eval
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from visbrain.objects import BrainObj, SceneObj, SourceObj
from matplotlib.colors import to_hex
from PIL import Image
from colorsys import hsv_to_rgb

## reading data

In [2]:
input_folder = '2_brain_visualization_preProcessing_1'
output_folder = '3_brain_visualization_preProcessing_2'

# Collect all CSV files from the input folder
csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

# Load data into DataFrames
df_patients = []
for file_name in csv_files:
    file_path = os.path.join(input_folder, file_name)
    df = pd.read_csv(file_path)
    df_patients.append(df)

# Generate a colormap for regions
area_per_patient = [df['area'].unique() for df in df_patients]
area_all_patients = pd.unique(np.concatenate(area_per_patient)).tolist()



num_colors = len(area_all_patients)

# high saturation prevents gray colors
colors = [
    hsv_to_rgb(i / num_colors, 0.90, 0.90)
    for i in range(num_colors)
]

region_colors = {
    region: to_hex(colors[i])
    for i, region in enumerate(area_all_patients)
}

## taking a look at areas

In [3]:
region_colors

{'TrIFG triangular part of the inferior frontal gyrus': '#e61717',
 'MFG middle frontal gyrus': '#e62c17',
 'MTG middle temporal gyrus': '#e64117',
 'ACgG anterior cingulate gyrus': '#e65617',
 'FO frontal operculum': '#e66b17',
 'LOrG lateral orbital gyrus': '#e68017',
 'Hippocampus': '#e69517',
 'Ent entorhinal area': '#e6aa17',
 'FuG fusiform gyrus': '#e6bf17',
 'PHG parahippocampal gyrus': '#e6d417',
 'AOrG anterior orbital gyrus': '#e2e617',
 'MCgG middle cingulate gyrus': '#cde617',
 'AIns anterior insula': '#b8e617',
 'Amygdala': '#a3e617',
 'Inf Lat Vent': '#8ee617',
 'OrIFG orbital part of the inferior frontal gyrus': '#79e617',
 'SFG superior frontal gyrus': '#64e617',
 'MOrG medial orbital gyrus': '#4fe617',
 'ITG inferior temporal gyrus': '#3ae617',
 'LiG lingual gyrus': '#25e617',
 'POrG posterior orbital gyrus': '#17e61e',
 'PP planum polare': '#17e633',
 'STG superior temporal gyrus': '#17e648',
 'CO central operculum': '#17e65d',
 'PrG precentral gyrus': '#17e672',
 'MS

## preprocessing 2:
I want to take a couple of brain areas ONLY and save those areas for each patient.

In [4]:
df_patients[0].head()

,Channel_Name,Nmm_atlas,area,coordinate_x,coordinate_y,coordinate_z
0,ROFC7,Right TrIFG triangular part of the inferior fr...,TrIFG triangular part of the inferior frontal ...,37.716870,32.353153,13.918516
1,LOFC10,Left MFG middle frontal gyrus,MFG middle frontal gyrus,-34.614600,33.788637,28.812653
2,RCD10,Right MFG middle frontal gyrus,MFG middle frontal gyrus,49.315260,21.861680,33.689576
3,LHIP9,Left MTG middle temporal gyrus,MTG middle temporal gyrus,-63.272704,-14.200921,-18.179159
4,ROFC8,Right MFG middle frontal gyrus,MFG middle frontal gyrus,38.234987,30.466426,19.424250


In [5]:


input_folder = '2_brain_visualization_preProcessing_1'
output_folder = '3_brain_visualization_preProcessing_2'
os.makedirs(output_folder, exist_ok=True)


target_areas = [
    "Amygdala",
    "AOrG anterior orbital gyrus",
    "Caudate",
    "Ent entorhinal area",
    "FO frontal operculum",
    "FuG fusiform gyrus",
    "Hippocampus",
    "ITG inferior temporal gyrus",
    "LOrG lateral orbital gyrus",
    "MCgG middle cingulate gyrus",
    "MTG middle temporal gyrus",
    "PHG parahippocampal gyrus",
    "PP planum polare",
    "PrG precentral gyrus",
    "SFG superior frontal gyrus",
    "TMP temporal pole",
    "Thalamus Proper",
    "TTG transverse temporal gyrus"
]


# create bright, highly saturated colors
# saturation=1 prevents gray or washed-out colors
num_colors = len(target_areas)

colors = [
    hsv_to_rgb((i / num_colors + 0.03) % 1.0, 1.0, 0.90)
    for i in range(num_colors)
]

region_colors = {
    area: to_hex(colors[i])
    for i, area in enumerate(target_areas)
}

print(region_colors)


csv_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith('.csv')
]


for file_name in csv_files:

    file_path = os.path.join(input_folder, file_name)
    df = pd.read_csv(file_path)

    if 'area' not in df.columns:
        print(f"Column 'area' not found in {file_name}. Skipping file.")
        continue

    # clean area names so that the color mapping does not fail
    df['area'] = df['area'].astype(str).str.strip()

    filtered_df = df[df['area'].isin(target_areas)].copy()

    if filtered_df.empty:
        print(f"Filtered DataFrame for {file_name} is empty. Skipping save.")
        continue

    filtered_df['left'] = 0
    filtered_df['right'] = 0

    if 'Nmm_atlas' in filtered_df.columns:

        atlas_text = filtered_df['Nmm_atlas'].fillna('').astype(str)

        filtered_df.loc[
            atlas_text.str.contains('left', case=False),
            'left'
        ] = 1

        filtered_df.loc[
            atlas_text.str.contains('right', case=False),
            'right'
        ] = 1

    # assign the actual color to every row
    filtered_df['color'] = filtered_df['area'].map(region_colors)

    # check that no area failed to receive a color
    if filtered_df['color'].isna().any():
        missing_areas = filtered_df.loc[
            filtered_df['color'].isna(), 'area'
        ].unique()

        print(f"Missing colors for: {missing_areas}")

        # bright magenta fallback instead of gray
        filtered_df['color'] = filtered_df['color'].fillna('#ff00ff')

    output_path = os.path.join(output_folder, file_name)
    filtered_df.to_csv(output_path, index=False)

    print(f"Filtered data saved to: {output_path}")

{'Amygdala': '#e62900', 'AOrG anterior orbital gyrus': '#e67600', 'Caudate': '#e6c200', 'Ent entorhinal area': '#bce600', 'FO frontal operculum': '#70e600', 'FuG fusiform gyrus': '#23e600', 'Hippocampus': '#00e629', 'ITG inferior temporal gyrus': '#00e676', 'LOrG lateral orbital gyrus': '#00e6c2', 'MCgG middle cingulate gyrus': '#00bce6', 'MTG middle temporal gyrus': '#0070e6', 'PHG parahippocampal gyrus': '#0023e6', 'PP planum polare': '#2900e6', 'PrG precentral gyrus': '#7600e6', 'SFG superior frontal gyrus': '#c200e6', 'TMP temporal pole': '#e600bc', 'Thalamus Proper': '#e60070', 'TTG transverse temporal gyrus': '#e60023'}
Filtered data saved to: 3_brain_visualization_preProcessing_2\201810.csv
Filtered data saved to: 3_brain_visualization_preProcessing_2\201811.csv
Filtered data saved to: 3_brain_visualization_preProcessing_2\201901.csv
Filtered data saved to: 3_brain_visualization_preProcessing_2\201902.csv
Filtered data saved to: 3_brain_visualization_preProcessing_2\201902r.csv
